
# <center> **Global Markets,Investments & Analytics**


## <center> BUS 520 | MBA 
### <center> Becamex Business School

### <center> Kang Seung Won (kwon.kang@eiu.edu.vn)
##### <center> Miss Ninh Thi Trang (trang.ninh@eiu.edu.vn), Miss Do Thanh Thuy Vy (vy.do@eiu.edu.vn), Mr. Lam Truong Son (son.lam@eiu.edu.vn)

---
## <center> **Week 2 (Course Week 8): Financial Analysis & Customer Analytics**

---
## Learning Objectives: Week 2 (Course Week 8)

**Main Question:** How can managers use real-world data to make better decisions?

1. **Review:** Review key concepts from the previous week.

2. **Regression Analysis:** Estimate and interpret relationships between variables using Ordinary Least Squares (OLS) and Logistic Regression (Logit).

3. **Financial Performance and Risk:** Calculate and interpret daily returns, volatility, the Sharpe ratio, and beta, and apply the Capital Asset Pricing Model (CAPM).

4. **Customer Analytics:** Calculate key sales performance indicators (KPIs) using real-world UK retail transaction data and conduct RFM (Recency, Frequency, and Monetary Value) analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
sns.set_style("whitegrid")

mpl.rcParams['figure.figsize'] = (12, 6)
mpl.rcParams['axes.grid'] = True

import warnings
warnings.filterwarnings("ignore")

### Review: From Sample to Population

1. **Inference about the Population Mean:** Using the t-distribution.
2. **Optional: Inference about the Population Variance:** Using the chi-square ($\chi^2$) distribution.

In [ ]:
from faker import Faker

np.random.seed(42)
N = 3000

s1 = np.clip(np.random.normal(8.7, 0.9, N), 4, 10)
s2 = np.clip(np.random.normal(8.6, 0.9, N), 4, 10)
s3 = np.clip(np.random.normal(8.7, 0.9, N), 4, 10)
total = s1 + s2 + s3

total = total * (28.0 / total.mean())
ratio = total / (s1+s2+s3)
s1, s2, s3 = s1*ratio, s2*ratio, s3*ratio

base = 1.0 + 3.0 * np.random.beta(6, 3, N)  # mean 3.0 exactly
gpa = base + 0.05 * (total - 28) + np.random.normal(0, 0.15, N)
gpa = np.clip(gpa, 1.0, 4.0).round(2)

s1 = np.clip(s1, 0, 10).round(1)
s2 = np.clip(s2, 0, 10).round(1)
s3 = np.clip(s3, 0, 10).round(1)
total = (s1+s2+s3).round(1)

EIU_population = pd.DataFrame({
    "student_id": [f"{24}{i+1:04d}" for i in range(N)],
    "exam_s1": s1,
    "exam_s2": s2,
    "exam_s3": s3,
    "exam_total": total,
    "gpa": gpa
})
EIU_population

In [ ]:
rng = np.random.RandomState(456) 
gpa = EIU_population['gpa']
N = 30  
sample_gpa = rng.choice(gpa, size=N, replace=False)
sample_gpa

In [ ]:
sample_gpa_mean = sample_gpa.mean()
print('sample mean:', sample_gpa_mean)
sample_gpa_var = sample_gpa.var(ddof=1)
print('sample variance:', sample_gpa_var)
sample_gpa_sd = sample_gpa.std(ddof=1)
print('sample standard deviation:', sample_gpa_sd)

---
> **Could the population mean GPA be 3.5?**

This claim is the **null hypothesis**:

$$H_0:\mu=3.5$$

We know the sample mean. We estimate SE using the **sample SD** ($s$) instead of the population SD ($\sigma$):

$$SE_{\mathrm{estimated}}=\frac{s}{\sqrt n}$$

Then we calculate a **t-score**, not a Z-score:

$$t=\frac{\bar{x}-3.5}{s/\sqrt n}$$

In [ ]:
t_score = (sample_gpa.mean() - 3.5) / (sample_gpa.std(ddof=1) / np.sqrt(N))
t_score

In [ ]:
from scipy import stats

p_value = 2 * stats.t.sf(abs(t_score), df=N-1) # sf: survival function
print('p value:', p_value)

In [ ]:
stats.ttest_1samp(sample_gpa, popmean=3.5)

In [ ]:
p_value = stats.ttest_1samp(sample_gpa, popmean=3.5).pvalue
p_value

### Confidence Interval

A confidence interval goes one step further than giving only a sample average.

For a mean, a common **95% t confidence interval** is:

$$\bar{x}\;\pm\;t_{0.975,\;n-1}\times\frac{s}{\sqrt n}$$


> **Sample mean ± (critical t-value × estimated SE)**


If we repeated the sampling process many times and made one interval per sample, about **95% of those intervals** would include the true population mean, when the statistical assumptions hold.

In [ ]:
t_ci = stats.t.ppf(0.975, df=N-1) # ppf: percent point function (left-tail probability)
print(t_ci)
lower = sample_gpa.mean() - t_ci * sample_gpa.std(ddof=1) / np.sqrt(N)
upper = sample_gpa.mean() + t_ci * sample_gpa.std(ddof=1)/ np.sqrt(N)
print(f"95% CI: ({lower:.4f}, {upper:.4f})")

In [ ]:
stats.t.interval(confidence=0.95, df=len(sample_gpa) - 1, loc=sample_gpa.mean(), scale=stats.sem(sample_gpa))

### Optional: Inference about the Population Variance

When the population is **normally distributed**, the sampling
distribution of the sample variance follows a scaled
chi-square distribution:

$$
\frac{(n-1)s^2}{\sigma^2} \sim \chi^2_{n-1}
$$

The $(1-\alpha)\times100\%$ confidence interval for the
population variance is:

$$
\left[
\frac{(n-1)s^2}{\chi^2_{1-\alpha/2,n-1}},
\frac{(n-1)s^2}{\chi^2_{\alpha/2,n-1}}
\right]
$$

where:

- $n$: Sample size
- $s^2$: Sample variance
- $\sigma^2$: Population variance
- $n-1$: Degrees of freedom
- $\alpha$: Significance level

**Assumptions:** The observations are independently sampled
from a normally distributed population.

**Managerial implication:** Estimating population variance
helps managers assess uncertainty, operational consistency,
financial risk, and variability in business performance.

In [ ]:
# Histogram
plt.hist(sample_gpa, bins=10, edgecolor='black')
plt.title('Distribution of Sample GPA')
plt.xlabel('GPA')
plt.ylabel('Frequency')
plt.show()

# Shapiro-Wilk normality test
stat, p_value = stats.shapiro(sample_gpa)

print(f"Shapiro-Wilk statistic: {stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject H0: Evidence against population normality.")
else:
    print("Fail to reject H0")

In [ ]:
s2 = np.var(sample_gpa, ddof=1)

# Hypothesized population variance
sigma2_0 = 0.25
alpha = 0.05
df = N - 1

# Chi-square test statistic
chi2_stat = (N - 1) * s2 / sigma2_0

# Two-tailed p-value
p_value = 2 * min(
    stats.chi2.cdf(chi2_stat, df),
    stats.chi2.sf(chi2_stat, df)
)

print(f"Sample variance: {s2:.4f}")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {df}")
print(f"p-value: {p_value:.4f}")

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

---
# From Sample to Population: Regression and Classification


## The goal

We just learned how to use a **sample mean**, its **SE**, and a **p-value** to ask a question about a **population mean**.

Now we will do the same for **relationships between students**:

| Method | X (inputs) | Y (outcome) | Question |
|---|---|---|---|
| **Linear regression (OLS)** | `exam_total` | `gpa` | How is entrance-exam score related to GPA? |
| **Logistic regression (Logit)** | `gpa`, `exam_total` | `ielts_pass` = 0 or 1 | Do these scores help explain whether a student gets an IELTS score **above 6.0**? |



In [ ]:
# Keep the original 3,000 students. Add invented IELTS data.
from datetime import date

fake = Faker("vi_VN")
Faker.seed(2026)
rng_ielts = np.random.default_rng(2026)

EIU_population = EIU_population.copy()
student_count = len(EIU_population)

EIU_population["student_name"] = [fake.name() for _ in range(student_count)]
EIU_population["ielts_exam_date"] = [
    fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 10, 31))
    for _ in range(student_count)
]
EIU_population["ielts_study_months"] = rng_ielts.integers(4, 21, student_count)

# The score is invented for teaching, with random variation.
raw_score = (5.90
             + 0.55 * (EIU_population["gpa"] - 3.0)
             + 0.16 * (EIU_population["exam_total"] - 28.0)
             + rng_ielts.normal(0, 0.65, student_count))

# IELTS is reported in half-band steps; keep scores between 0 and 9.
EIU_population["ielts_score"] = np.clip(np.round(raw_score * 2) / 2, 0, 9)
EIU_population["ielts_pass"] = (EIU_population["ielts_score"] > 6.0).astype(int)

# Extra label for our earlier one-year discussion (NOT used as Logit Y).
EIU_population["ielts_pass_within_1year"] = (
    (EIU_population["ielts_score"] > 6.0)
    & (EIU_population["ielts_study_months"] <= 12)
).astype(int)

print("Invented population size:", len(EIU_population))
print("IELTS pass rate (score > 6):", f'{EIU_population["ielts_pass"].mean():.1%}')
display(EIU_population[["student_name", "exam_total", "gpa",
                        "ielts_score", "ielts_pass",
                        "ielts_study_months", "ielts_pass_within_1year"]].head())


In [ ]:
student_sample = EIU_population.sample(n=300, random_state=2026).copy()

print("Population size:", len(EIU_population))
print("Sample size:", len(student_sample))
print("Sample GPA mean:", round(student_sample["gpa"].mean(), 3))
print("Sample IELTS pass rate:", f'{student_sample["ielts_pass"].mean():.1%}')
print(student_sample["ielts_pass"].value_counts().sort_index())


In [ ]:
student_sample


| Method | X (inputs) | Y (outcome) | Question |
|---|---|---|---|
| **Linear regression (OLS)** | `exam_total` | `gpa` | How is entrance-exam score related to GPA? |**

##### Calculation of OLS Coefficients
### $$ y = X\beta + \epsilon $$

- **Sum of Squared Residuals (SSR)**: $ SSR = \epsilon'\epsilon = (y - X\beta)'(y - X\beta) $
- Expand SSR: $ SSR = y'y - \beta'X'y - y'X\beta + \beta'X'X\beta $
- (Since $y'X\beta$ is scalar, $y'X\beta = \beta'X'y$)
- Simplify, $ SSR = y'y - 2\beta'X'y + \beta'X'X\beta $
- Differentiate w.r.t. $\beta$, $ \frac{\partial SSR}{\partial \beta} = -2X'y + 2X'X\beta $
- Set Derivative to Zero (Normal Equations)
$ -2X'y + 2X'X\beta = 0 \implies X'X\beta = X'y $
- Solve for $\beta$, 

$$
\hat{\beta}_{OLS} = \underset{\beta}{\text{argmin}} \sum_{i=1}^{n} (y_i - x_i\beta)^2 = (X'X)^{-1}X'y
$$


In [ ]:
import statsmodels.api as sm
import pingouin as pg

In [ ]:
x_ols = student_sample['exam_total']
x_ols = sm.add_constant(student_sample['exam_total'])
x_ols

In [ ]:
y_ols = student_sample['gpa']

In [ ]:
beta_manual = np.linalg.inv(x_ols.T @ x_ols)@x_ols.T @ y_ols
beta_manual

In [ ]:
ols = sm.OLS(y_ols, x_ols).fit()
print(ols.summary())

In [ ]:
X_pop = sm.add_constant(EIU_population[["exam_total"]])
y_pop = EIU_population["gpa"]

ols_pop = sm.OLS(y_pop, X_pop).fit()

population_beta = ols_pop.params["exam_total"]
population_beta

In [ ]:
N = 300
tries = 1000
betas = []

for i in range(tries):

    student_sample = EIU_population.sample(n=N, replace=True)
    X_ols = sm.add_constant(student_sample[["exam_total"]])
    y_ols = student_sample["gpa"]

    ols = sm.OLS(y_ols, X_ols).fit()
    betas.append(ols.params["exam_total"])

betas = np.array(betas)

In [ ]:
sns.histplot(betas, bins=30, kde=True, stat="density")

plt.axvline(betas.mean(), color="blue",
            linestyle="--", label="Mean of Estimated Betas")

plt.title(f"Sampling Distribution of OLS Beta (N={N})")
plt.xlabel("Estimated Beta")
plt.legend()
plt.show()

## From a Sample to a POPULATION

This is the **same idea** as sampling means. If we took another random sample of 300 students, our fitted slope would change.

- **Sample slope $b_1$:** our best estimate from these students.
- **Population slope $\beta_1$:** the relationship we want to learn about for all 3,000 students.
- **SE of the slope:** how much the slope estimate would vary across samples (approximately).
- **95% confidence interval:** a range of possible population slopes compatible with our data and model.
- **p-value:** how strongly the sample contradicts a proposed **zero population slope**.

This **SE** is *not* the earlier `SD / sqrt(n)` for a sample mean. The regression program calculates the **coefficient's own SE**, considering the scatter around the line and spread of X values.


### Calculate the standard error of the OLS slope

Our fitted line for the **240 students** is $\widehat{GPA}_i=b_0+b_1\,exam\_total_i$. The **population slope** $\beta_1$ is fixed but unknown; the **sample slope** $b_1$ (also written $\hat\beta_1$) changes across random samples. We estimate the standard deviation of that sampling distribution with $SE(b_1)$.

For **simple OLS with an intercept and one predictor** (`exam_total`), the usual textbook SE is:

$$\boxed{SE(b_1)=\sqrt{\frac{s^2}{S_{xx}}}=\frac{s}{\sqrt{\sum_{i=1}^{n}(x_i-\bar x)^2}}},\qquad S_{xx}=\sum_{i=1}^{n}(x_i-\bar x)^2.$$

Follow the five steps below **using our actual sample**:

1. **Predict GPA for each student.** $\hat y_i=b_0+b_1x_i$.
2. **Find each residual.** $e_i=y_i-\hat y_i$ is the vertical distance from a student's observed GPA to the fitted line.
3. **Measure unexplained GPA variation.** Add up squared residuals, $SSE=\sum e_i^2$, then estimate error variance $s^2=\dfrac{SSE}{n-2}$. We divide by **$n-2$** because we estimated **two coefficients** (intercept and slope). Its square root, $s$, is the *residual standard error* in GPA units.
4. **Measure the spread of exam scores.** $S_{xx}=\sum(x_i-\bar x)^2$; a wider range of X values makes the slope easier to estimate precisely.
5. **Combine the two parts.** $SE(b_1)=\sqrt{\dfrac{SSE/(n-2)}{S_{xx}}}$. Larger residual scatter **increases** SE; wider X spread **decreases** SE, all else equal.

The regression package computes this for us; next we reproduce it **manually** and check that it matches `ols.bse["exam_total"]`. Note that this is **not** the sample-mean formula `SD / sqrt(n)`.


## Why is there a p-value for a regression slope?

Ask a specific question about the **population**:

$$H_0: \beta_1=0\qquad H_1: \beta_1\ne0$$

- **$H_0$:** in the population, there is no *linear* association between exam total and GPA (zero slope).
- **$H_1$:** the population slope is not zero.


In [ ]:
ols.params['exam_total']
ols.bse['exam_total']

In [ ]:
t_score = (ols.params['exam_total'] - 0) / ols.bse['exam_total']
t_score

In [ ]:
# p-value
p_value = 2 * stats.t.sf(abs(t_score), df=len(x_ols)-2)
p_value

In [ ]:
ols.tvalues['exam_total']
ols.pvalues['exam_total']
ols.conf_int(alpha=0.05).loc['exam_total']

---
## Linear Regression with Real Data: House Prices 
### House prices (`hprice1`)

**Business question:** How much is an additional square foot of house space associated with the selling price, after accounting for lot size and bedrooms?

88 houses sold in the Boston area in 1990.

| Variable | Meaning |
|---|---|
| `price` (Y) | Selling price (thousand US dollars) |
| `sqrft` (X1) | House size (square feet) |
| `lotsize` (X2) | Lot size (square feet) |
| `bdrms` (X3) | Number of bedrooms |

[Download CSV](https://vincentarelbundock.github.io/Rdatasets/csv/wooldridge/hprice1.csv) · [Variable descriptions](https://justinmshea.github.io/wooldridge/reference/hprice1.html)

**Data access:** The following code downloads CSV files directly. If classroom Wi-Fi is unavailable, download them beforehand and place `hprice1.csv`, `ceosal1.csv` and `coke.csv` in the notebook's working directory. No special dataset package is needed.



In [ ]:
hprice1 = pd.read_csv("https://vincentarelbundock.github.io/Rdatasets/csv/wooldridge/hprice1.csv")
hprice1

### Simple regression: price and house size

$$price_i=\beta_0+\beta_1 sqrft_i+u_i$$


In [ ]:
house_data = hprice1[["price", "sqrft", "lotsize", "bdrms"]].dropna()
house_simple = sm.OLS(house_data["price"], sm.add_constant(house_data[["sqrft"]])).fit()
print(house_simple.summary())


---
### Example 2. CEO compensation (`ceosal1`)

**Business question:** Are CEOs of larger or more profitable firms paid more?

209 CEOs, historical US company data (1990 salaries and sales).

| Variable | Meaning |
|---|---|
| `salary` (Y) | CEO annual salary (thousand USD) |
| `sales` (X1) | Firm sales (million USD) |
| `roe` (X2) | Return on equity (percentage points, 1988–90 average) |
| `ros` (X3) | Return on firm's stock (1988–90) |

[Download CSV](https://vincentarelbundock.github.io/Rdatasets/csv/wooldridge/ceosal1.csv) · [Variable descriptions](https://justinmshea.github.io/wooldridge/reference/ceosal1.html)

In [ ]:
ceo_data = pd.read_csv("https://vincentarelbundock.github.io/Rdatasets/csv/wooldridge/ceosal1.csv")
ceo_data

### Estimate the relationship

$$salary_i=\beta_0+\beta_1 sales_i+\beta_2 roe_i+\beta_3 ros_i+u_i$$

In [ ]:
ceo_model = sm.OLS(ceo_data["salary"], sm.add_constant(ceo_data[["sales", "roe", "ros"]])).fit(cov_type="HC3")
print(ceo_model.summary())

---
## **Financial Performance and Risk**
### Business case: Should an investor choose VinFast or a US market fund?

- **VinFast (VFS):** A Vietnamese electric vehicle manufacturer listed on Nasdaq on August 15, 2023.
- **SPY:** An exchange-traded fund (ETF) that tracks the S&P 500 Index.

**Source pages:**
- Yahoo Finance, VFS: https://finance.yahoo.com/quote/VFS/history/
- Yahoo Finance, SPY: https://finance.yahoo.com/quote/SPY/history/
- FRED, 3-month Treasury yield (`DGS3MO`): https://fred.stlouisfed.org/series/DGS3MO


In [ ]:
import yfinance as yf

tickers = ['VFS', 'SPY']
prices = yf.download(tickers, start="2023-08-15", auto_adjust=True, progress=False)['Close'].dropna()
prices

In [ ]:
prices.plot();

In [ ]:
prices.describe()

In [ ]:
prices.apply(lambda x: (x - x.mean()) / x.std()).plot();
prices.last('2Y').apply(lambda x: (x - x.mean()) / x.std()).plot();
prices.last('1Y').apply(lambda x: (x - x.mean()) / x.std()).plot();


## Understanding Data: 

In [ ]:
prices.describe()

---
### 1) Stock return

**Daily return:** How much did the price change since the previous trading day?

$$r_t = (P_t - P_{t-1}) / P_{t-1} = P_t/P_{t-1}-1$$

**Total return over our sample:** Compare the last adjusted price with the first price.

$$R = P_{end}/P_{start}-1$$

In [ ]:
d_return = (prices / prices.shift(1)) - 1
d_return

In [ ]:
prices.pct_change()

In [ ]:
returns = prices.pct_change().dropna()
total_return = prices.iloc[-1] / prices.iloc[0] - 1
print("Total return over the selected period:", (total_return * 100).round(2))

In [ ]:
returns

---
### 2) Volatility

- **Volatility:** How much daily returns change around their average.
- **Annualized volatility:** Daily standard deviation × $\sqrt{252}$.

In [ ]:
returns.plot();

fig, axes = plt.subplots(2, 1)

returns['VFS'].plot(ax=axes[0], title='Vinfast')
returns['SPY'].plot(ax=axes[1], title='SPY 500')
plt.show();


In [ ]:
annual_volatility = returns.std() * np.sqrt(252)

risk_table = pd.DataFrame({
    "Annual volatility (%)": annual_volatility * 100,
})
risk_table.round(2)


---
### 3) Sharpe Ratio: return above a low-risk rate

**Question:** How much extra return did the investor get for the return volatility?

- For this class, we use the **US 3-month Treasury constant-maturity yield** as a *proxy* for the risk-free rate.
- FRED series: **DGS(daily government security)3MO**. It is an **annual percentage yield**, not a daily return.

| Treasury Maturity | Common Use |
|---|---|
| 3-month (`DGS3MO`) | Risk-free rate for Sharpe ratio and short-term investment analysis |
| 2-year (`DGS2`) | Monetary policy expectations and interest-rate analysis |
| 5-year (`DGS5`) | Medium-term interest rates and bond market analysis |
| 10-year (`DGS10`) | Long-term interest rates, macroeconomic analysis, and valuation |
| 30-year (`DGS30`) | Long-term financing, pensions, and duration analysis |

**Steps:**
1. Read historical Treasury yields from FRED.
2. Convert each annual yield to a one-day rate.
3. Subtract the daily low-risk rate from each daily stock return.
4. Divide average daily excess return by daily volatility; multiply by $\sqrt{252}$.

$$Sharpe = \frac{\overline{r-r_f}}{s(r)}\sqrt{252}$$




In [ ]:
# FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO"
# FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS5"
fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO")
fred

In [ ]:
fred.isnull().sum()

In [ ]:
fred.info()
fred.dtypes
fred.describe()

In [ ]:
fred.columns = ["Date", "AnnualYieldPercent"]
fred

In [ ]:
fred["Date"] = pd.to_datetime(fred["Date"]) 
fred["AnnualYieldPercent"] = fred["AnnualYieldPercent"].ffill()

# Match with stock return dates
rf = fred.set_index("Date")["AnnualYieldPercent"]
rf = rf.reindex(returns.index, method="ffill")

# Convert annual percentage yield to daily rate
rf_daily = rf / 100 / 252

In [ ]:
rf
rf_daily

In [ ]:
data = returns.copy()
data["RF_Daily"] = rf_daily

data.head(10)

In [ ]:
# Daily stock returns
stock_returns = data[["SPY", "VFS"]]

# Daily excess returns (stock return - risk-free rate)
excess_daily = stock_returns.sub(data["RF_Daily"], axis=0) # axis=0: row-wise, axis=1: column wise
excess_daily 

# Annualized Sharpe Ratio
sharpe_ratio = ((excess_daily.mean()/ stock_returns.std())* np.sqrt(252))

print("Sharpe Ratio:")
display(sharpe_ratio.round(3).rename("Sharpe Ratio").to_frame())

### Meaning: Sharpe Ratio?

- **Positive:** Average return was above the low-risk rate, after accounting for volatility.
- **Negative:** Average return was below the low-risk rate.
- **Higher:** More average excess return per unit of volatility *in this sample*.


---
### 4) CAPM (Captital Asset Pricing Model) and beta


CAPM estimates the expected return of an investment based on its systematic (market) risk.

$$
\boxed{E(R_i)=R_f+\beta_i[E(R_m)-R_f]}
$$

where:

- $E(R_i)$: Expected return of asset $i$
- $R_f$: Risk-free rate
- $\beta_i$: Sensitivity of asset $i$ to market movements
- $E(R_m)$: Expected market return
- $E(R_m)-R_f$: Market risk premium

**Main Question:** What return should an investor expect as compensation for taking systematic risk?

- **VFS:** Individual asset under consideration.
- **SPY:** Proxy for the market portfolio.
- **US Treasury Bill Rate:** Proxy for the risk-free rate.

We already calculated daily returns. Check two columns:
- **X:** SPY daily return (market proxy: about 80% of US market).
- **Y:** VFS daily return (stock).

Both are decimal returns in Python. Multiply by **100** to show percent.

**Important:** SPY is a US stock-market proxy. It is not the full world market.

In [ ]:
(returns * 100).round(2)

In [ ]:
returns.plot();

### Step 2. Estimate beta with OLS regression

We use the same regression method as in Week 1:

$$R_{VFS,t}=a+\beta R_{SPY,t}+e_t$$

- **a (intercept):** Fitted VFS return when SPY return is zero.
- **Beta (slope):** Change in the fitted VFS return for a one-unit change in SPY return.
- **e (error):** Part of VFS's return not explained by this fitted line.

**Example:** If beta is 1.5, a **1 percentage-point** higher SPY daily return is linked to a **1.5 percentage-point** higher fitted VFS daily return.
This is an average relationship in the data. It is not a promise about tomorrow.

Beta can also be calculated without regression:

$$\beta=\frac{Cov(R_{VFS},R_{SPY})}{Var(R_{SPY})}$$

**Task:** Run OLS, find beta, and check that both methods give the same number.

In [ ]:
# OLS: X is SPY daily return; Y is VFS daily return.
import statsmodels.api as sm

X_beta = sm.add_constant(returns["SPY"])  
y_beta = returns["VFS"]
beta_model = sm.OLS(y_beta, X_beta).fit()

print(beta_model.summary())

In [ ]:
import pingouin as pg
ols_pg = pg.linear_regression(returns["SPY"], returns["VFS"])
ols_pg

In [ ]:
beta = np.linalg.inv(X_beta.T@ X_beta )@ X_beta.T @ y_beta
beta

In [ ]:

beta_from_covariance = returns["VFS"].cov(returns["SPY"]) / returns["SPY"].var()
beta_from_covariance

In [ ]:

beta_vfs = float(beta_model.params["SPY"])
intercept_daily = float(beta_model.params["const"])
r_squared = float(beta_model.rsquared)

print("OLS beta (VFS vs SPY):", round(beta_vfs, 3))
print("Covariance-variance beta:", round(beta_from_covariance, 3))
print("R-squared:", round(r_squared, 3))


In [ ]:
sns.regplot(x='SPY', y='VFS', data=returns);

---
### Step 3. Use beta to calculate CAPM required return

$$E(R_{VFS})=R_f+\beta_{VFS}\left[E(R_m)-R_f\right]$$

- **$R_f$:** Annual low-risk rate. We use the latest matched FRED 3-month US Treasury yield as a proxy.
- **$\beta_{VFS}$:** Beta from Step 2.
- **$E(R_m)-R_f$:** Expected annual market risk premium. 
- **$E(R_{VFS})$:** CAPM **required annual return**, based on these inputs.


---
# Logistic Regression (Logit): GPA + Exam Total → IELTS Pass 

## Why is this classification and not linear regression?

Here the outcome is not a GPA number. It is **yes/no**:

- **Y = `ielts_pass` = 1** if invented IELTS score **> 6.0**.
- **Y = `ielts_pass` = 0** if invented IELTS score **≤ 6.0**.

The predictors are **X = GPA and entrance-exam total score**. Do **not** put `ielts_score` into the predictors; it directly tells us the answer and would cause **data leakage**. We also do not include `ielts_pass_within_1year`.

**Logistic regression (Logit)** predicts a probability between **0% and 100%**. It is a regression *model* used for **binary classification**.


### Why take the log of odds? (Logit in easy steps)

Linear regression predicts a number using a straight line. But when Y is 0 or 1, a straight line can predict below 0 or above 1. We need a way to convert **any real-valued score** into a valid probability.

**1. Probability → Odds → Log-odds**



Let $p=P(Y=1\mid X)$, the probability of an event (e.g., IELTS pass).

$$\text{Odds}=\frac{p}{1-p},\qquad \text{Logit}=\log\left(\frac{p}{1-p}\right)$$


**2. Linear combination (the same starting idea as OLS)**

$$z=w_0+w_1x_1+\cdots+w_kx_k=\mathbf{w}^{T}\mathbf{X}$$

Here the vector $\mathbf{X}=(1,x_1,\ldots,x_k)^T$ includes a 1 for the intercept. Logistic regression sets

$$\log\left(\frac{p}{1-p}\right)=z.$$

When $x_j$ increases by one unit, log-odds change by $w_j$, **holding all other predictors fixed**. Odds are multiplied by $e^{w_j}$. Probability does **not** change by a fixed amount.

**3. Solve for probability: the sigmoid function**

Take the exponential of both sides and rearrange:

$$\frac{p}{1-p}=e^z$$
$$p=e^z(1-p)$$
$$p(1+e^z)=e^z$$
$$\boxed{p=\frac{e^z}{1+e^z}=\frac{1}{1+e^{-z}}=\sigma(z)}$$

This S-shaped **sigmoid** converts any real score $z$ into a probability between 0 and 1. When $z=0$, $p=0.5$; negative scores give probabilities below 0.5 and positive scores above 0.5.

---
# Logistic Regression with Real Data: Coke or Pepsi?


The `coke` dataset from *Principles of Econometrics* contains scanner data on 1,140 individuals who bought Coke or Pepsi. It is historical teaching data, not a current market survey.

| Variable | Meaning |
|---|---|
| `coke` (Y) | 1 = Coke chosen; 0 = Pepsi chosen |
| `pratio` (X1) | Price of Coke / price of Pepsi |
| `disp_pepsi` (X2) | 1 = Pepsi on display; 0 = not displayed |
| `disp_coke` (X3) | 1 = Coke on display; 0 = not displayed |

**Expected signs:** A higher relative Coke price may reduce Coke purchase probability; a Coke display may increase it, while a Pepsi display may reduce it. We test these expectations using the data.

[Download CSV](https://www.principlesofeconometrics.com/poe5/data/csv/coke.csv) · [Variable descriptions](https://rdrr.io/github/ccolonescu/PoEdata/man/coke.html)

In [ ]:
coke = pd.read_csv("https://www.principlesofeconometrics.com/poe5/data/csv/coke.csv")
coke

## Estimation


$$\log\left(\frac{p}{1-p}\right)=\beta_0+\beta_1 pratio+\beta_2 disp\_pepsi+\beta_3 disp\_coke$$

where $p=P(coke=1\mid X)$.

---
# **Customer Analytics**


### Customer decisions: UCI Online Retail
- UK gift retailer, December 2010 to December 2011.
- Invoice lines, customers, and prices.
- Dataset: https://archive.ics.uci.edu/dataset/352/online+retail
- We use it only for sales KPIs and RFM customer groups.

## Online transaction data


In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
print(online_retail.metadata) 

print(online_retail.variables) 


In [ ]:
retail_raw = pd.concat([online_retail.data.ids, online_retail.data.features], axis=1)


In [ ]:
retail_raw

## Clean the data

An invoice can contain many rows. A row is **not** one order.

`InvoiceNo` starting with **C** means a cancellation.

In [ ]:
retail_raw.info()
retail_raw.isnull().sum()

In [ ]:
retail = retail_raw.copy()
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"])


In [ ]:
retail.info()

In [ ]:
contains_c = retail['InvoiceNo'].str.contains('C')
retail[contains_c]

In [ ]:
# canceled = retail[contains_c]
valid_sales = retail.loc[(~contains_c) & (retail["Quantity"] > 0) & (retail["UnitPrice"] > 0)].copy()
valid_sales["SalesGBP"] = valid_sales["Quantity"] * valid_sales["UnitPrice"] #  Gross Recorded Sales

In [ ]:
valid_sales.isna().sum()

In [ ]:
# For Member and Non-Member: 132220
customer_sales = valid_sales.dropna(subset=["CustomerID"]).copy()
customer_sales["CustomerID"] = customer_sales["CustomerID"].astype("int64").astype(str) # Int --> Str


In [ ]:
customer_sales.info()

In [ ]:

print("Original line items:", len(retail))
print("Positive sales line items:", len(valid_sales))
print("Line items with membership customer:", len(customer_sales))
print("Customer IDs missing from positive-sales rows (Non-Members):", valid_sales["CustomerID"].isna().sum())

---
## Calculate customer and sales KPIs

A **KPI** is a number that answers a business question.

| KPI | Simple definition |
|---|---|
| Positive recorded sales | Sum of `Quantity × UnitPrice` after the stated filters, in GBP |
| Orders | Number of distinct invoice numbers after filters |
| AOV | Positive recorded sales / orders, in GBP per order |
| Known customers | Number of distinct non-missing CustomerIDs |
| Repeat purchase rate | Known customers with 2+ invoices / known customers |



In [ ]:
positive_sales_gbp = valid_sales["SalesGBP"].sum()
order_count = valid_sales["InvoiceNo"].nunique()
avg_order_value = positive_sales_gbp / order_count
customer_order_counts = customer_sales.groupby("CustomerID")["InvoiceNo"].nunique()
known_customers = len(customer_order_counts)
repeat_rate = (customer_order_counts >= 2).mean()

customer_kpis = pd.DataFrame({
    "Metric": ["Positive recorded sales (GBP)", "Orders", "AOV (GBP/order)",
               "Known customers", "Repeat purchase rate (%)"],
    "Value": [positive_sales_gbp, order_count, avg_order_value,
              known_customers, repeat_rate * 100]
})
customer_kpis.round(2)

In [ ]:
# Group positive-sales amounts by calendar month.
monthly_sales = (valid_sales.set_index("InvoiceDate")["SalesGBP"]
                 .resample("MS").sum().rename("PositiveSalesGBP").to_frame())
monthly_sales.plot(title="Positive recorded sales by month", legend=False)
plt.ylabel("GBP")
plt.show()

In [ ]:
# Yearly sales
yearly_sales = (
    valid_sales.set_index("InvoiceDate")["SalesGBP"]
    .resample("YS").sum()
    .rename("PositiveSalesGBP").to_frame()
)

yearly_sales.plot(title="Positive recorded sales by year", legend=False)
plt.ylabel("GBP")
plt.show()

# Yearly sales
yearly_sales = (
    valid_sales.set_index("InvoiceDate")["SalesGBP"]
    .resample("YS").sum()
    .rename("PositiveSalesGBP").to_frame()
)

yearly_sales.plot(
    kind="bar",
    title="Positive recorded sales by year",
    legend=False,
    figsize=(8, 5)
)

plt.xlabel("Year")
plt.ylabel("Sales (GBP)")
plt.xticks(
    range(len(yearly_sales)),
    yearly_sales.index.year,
    rotation=0
)
plt.show()


# Weekly sales
weekly_sales = (
    valid_sales.set_index("InvoiceDate")["SalesGBP"]
    .resample("W-SUN").sum()
    .rename("PositiveSalesGBP").to_frame()
)

weekly_sales.plot(title="Positive recorded sales by week", legend=False)
plt.ylabel("GBP")
plt.show()

## RFM: put customers into groups

**RFM** means:
- **R — Recency:** Days since the most recent purchase. **Smaller** means more recent.
- **F — Frequency:** Number of different orders. **Larger** means more orders.
- **M — Monetary:** Total positive recorded sales. **Larger** means more spending.

We use the day **after the last observed sale** as the reference date.
We give a score of **1, 2, or 3** for each measure. **3 is better** on each scale.


In [ ]:
print(customer_sales["InvoiceDate"].max())

In [ ]:
reference_day = customer_sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

rfm = customer_sales.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    MonetaryGBP=("SalesGBP", "sum")
)
rfm["RecencyDays"] = (reference_day - rfm["LastPurchase"].dt.normalize()).dt.days

# Rank first to avoid errors when many customers have the same value.
rfm["R"] = pd.qcut(rfm["RecencyDays"].rank(method="first"), 3,
                   labels=[3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 3,
                   labels=[1, 2, 3]).astype(int)
rfm["M"] = pd.qcut(rfm["MonetaryGBP"].rank(method="first"), 3,
                   labels=[1, 2, 3]).astype(int)

rfm[["RecencyDays", "Frequency", "MonetaryGBP", "R", "F", "M"]]

### Define clear customer groups

- **Loyal high-value:** R, F, and M are all at least 2.
- **At risk:** R is 1, but F is at least 2. These customers bought before but not recently.
- **Recent one-time:** R is 3 and F is 1.
- **Other:** All remaining customers.

These are **our classroom rules**. Other businesses may need different rules.

In [ ]:
rfm["Segment"] = np.select(
    [
        (rfm["R"] >= 2) & (rfm["F"] >= 2) & (rfm["M"] >= 2),
        (rfm["R"] == 1) & (rfm["F"] >= 2),
        (rfm["R"] == 3) & (rfm["F"] == 1)
    ],
    ["Loyal high-value", "At risk", "Recent one-time"],
    default="Other"
)

segment_summary = (rfm.groupby("Segment")
                   .agg(Customers=("Frequency", "size"),
                        PositiveSalesGBP=("MonetaryGBP", "sum"),
                        MeanOrders=("Frequency", "mean"))
                   .sort_values("PositiveSalesGBP", ascending=False))
segment_summary["SalesSharePct"] = (
    100 * segment_summary["PositiveSalesGBP"] / rfm["MonetaryGBP"].sum()
)
display(segment_summary.round(2))
segment_summary["Customers"].plot.bar(title="Known customers by RFM group")
plt.ylabel("Customers")
plt.xticks(rotation=20, ha="right")
plt.show()